## IMPORTAÇÕES

In [ ]:
import pandas as pd
from pathlib import Path
from src.config import settings
from src.transform import reshape
import os

In [9]:
pd.options.display.float_format = '{:,.2f}'.format

## CARREGAMENTO DAS BASES

In [2]:
CENSO_PATH          = settings.EXTERNAL_DATA_PATH / 'censo_demografico_2022.csv'
CENSO_PCD_PATH      = settings.EXTERNAL_DATA_PATH / 'censo_demografico_pcd_2022.csv'
IBGE_TABELA         = settings.EXTERNAL_DATA_PATH / 'POP2024_20241230.xls'
GEO_JSON            = settings.EXTERNAL_DATA_PATH / 'br_states.json'
TABELA_SNIIC        = settings.FINAL_DATA_PATH / 'tabela_final_3_2_2026_14_22.parquet'



In [3]:
# IBGE - CENSO
df_censo        = pd.read_csv(CENSO_PATH, sep=';')
df_censo_pcd    = pd.read_csv(CENSO_PCD_PATH, sep=';')
df_censo_pcd.rename(columns={'Cód.':'cod_ibge', 'Total':'pop_pcd'}, inplace=True)

# IBGE - ESTADOS MUN
df_ibge_mun        = pd.read_excel(IBGE_TABELA, sheet_name='MUNICÍPIOS')
df_ibge_est        = pd.read_excel(IBGE_TABELA)

# SNIIC
df_sniic = pd.read_parquet(TABELA_SNIIC, engine='fastparquet')

## CENSO DEMOGRÁFICO

In [8]:
df_censo = df_censo.merge(
    right=df_censo_pcd[['cod_ibge', 'pop_pcd']],
    how='left',
    on='cod_ibge'   
)

In [9]:
# RENOMEIA AS COLUNAS DA TABELA ORIGEM
df_censo = df_censo.rename(columns=settings.RENAME_COLUMNS_CENSO)

In [10]:
# DEFINE COLUNA PESSOAS NEGRAS
df_censo["pop_pessoas_negras"] = df_censo["pop_preta"] + df_censo["pop_parda"]

In [11]:
# CRIA A COLUNA PERCENTUAL PARA CADA CATEGORIA DE POPULAÇÃO
df_censo[settings.COLS_NUM] = df_censo[settings.COLS_NUM].apply(
    pd.to_numeric, errors="coerce"
)

for col in settings.COLS_POP:
    df_censo[f"rel_{col}"] = df_censo[col] / df_censo["pop_total"]

In [12]:
# PASSA COD_IBGE PARA STRING, ESSA SERÁ NOSSA CHAVE PRIMÁRIA
df_censo['cod_ibge'] = df_censo['cod_ibge'].astype(str) 

## TABELA SNIIC

### ESTADOS

In [10]:
df_sniic_estados = df_sniic.loc[df_sniic['tipo_ente'] == 'ESTADO']

In [ ]:
# TRATAMENTO TABELA_SNIIC - CRIAMOS A TABELA DE VALORES AGREGADO
df_sniic_estados = reshape.agregar_tabela_sniic(df=df_sniic_estados)

In [ ]:
# CALCULO DE VALOR E VAGA
df_sniic_estados = reshape.calcular_totais_cotas(df=df_sniic_estados)

In [15]:
# CALCULA PROPORÇÃO VALOR VAGA
df_sniic_estados = reshape.calcular_proporcoes_valor_e_vagas(df=df_sniic_estados)

In [17]:
df_sniic_estados.columns

Index(['cod_ibge', 'ente_federativo', 'tipo_ente', 'sum_valor_total',
       'sum_valor_cotas_negras', 'sum_valor_cotas_indigenas',
       'sum_valor_cotas_pcd', 'sum_vagas_totais', 'sum_vagas_cotas_negras',
       'sum_vagas_cotas_indigenas', 'sum_vagas_cotas_pcd',
       'sum_valor_cotas_total', 'sum_vagas_cotas_total',
       'rel_valor_cotas_negras', 'rel_valor_cotas_indigenas',
       'rel_valor_cotas_pcd', 'rel_valor_cotas_total',
       'rel_vagas_cotas_negras', 'rel_vagas_cotas_indigenas',
       'rel_vagas_cotas_pcd', 'rel_vagas_cotas_totais'],
      dtype='str')

In [ ]:
df_sniic_estados[[
    'ente_federativo', 
    'sum_valor_cotas_negras', 
    'sum_valor_cotas_indigenas',
    'sum_valor_cotas_pcd',
    'rel_valor_cotas_negras', 
    'rel_valor_cotas_indigenas',
    'rel_valor_cotas_pcd'
]]